In [2]:
import numpy as np

code_space = []

class Error:
    def __init__(self, probability, type):
        self.probability = probability
        self.type = type

In [ ]:
class Circuit:
    def __init__(self, qubits):
        pass

In [ ]:
class RepititionCode:
    def __init__(self, distance):
        self.distance = distance

        self.qubits = np.zeros(distance)

    def x_error(self, component):
        if component > self.distance:
            raise IndexError

        self.qubits[component] = int(not self.qubits[component])


    def z_error(self, component):
        if component > self.distance:
            raise IndexError

In [80]:
dim = 5
R2 = 1 / np.sqrt(2)

def H(bit):

    had = np.array([
        [1*R2, 1*R2],
        [1*R2, -1*R2]])

    id = np.array([
        [1, 0],
        [0, 1]
    ])

    if dim == 1:
        return had

    chain = np.zeros(dim)
    chain[bit] = 1

    m0 = had if chain[0] == 1 else id
    m1 = had if chain[1] == 1 else id
    prod = np.kron(m0, m1)

    if dim == 2:
        return prod
    
    for i in chain[2:]:
        m = had if i == 1 else id
        prod = np.kron(prod, m)

    return prod


In [ ]:
def Z(bits: list):

    z = np.diag([1, -1])
    id = np.diag([1, 1])

    if dim == 1:
        return z

    chain = np.zeros(dim)
    chain[bits] = 1

    m0 = z if chain[0] == 1 else id
    m1 = z if chain[1] == 1 else id
    prod = np.kron(m0, m1)

    if dim == 2:
        return prod
    
    for i in chain[2:]:
        m = z if i == 1 else id
        prod = np.kron(prod, m)

    return prod

def X(bits: list):

    x = np.array([
        [0, 1],
        [1, 0]
    ])
    id = np.diag([1, 1])

    if dim == 1:
        return x

    chain = np.zeros(dim)
    chain[bits] = 1

    m0 = x if chain[0] == 1 else id
    m1 = x if chain[1] == 1 else id
    prod = np.kron(m0, m1)

    if dim == 2:
        return prod
    
    for i in chain[2:]:
        m = x if i == 1 else id
        prod = np.kron(prod, m)

    return prod


array([[0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 1],
       [1, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0]])

In [ ]:
dim = 5

def CNOT(control, target):
    x = np.array(
            [[0, 1],
            [1, 0]]
        )
    id = np.diag([1, 1])

    m = np.zeros([2**dim, 2**dim])

    for state in range(2**dim):
        binary = np.zeros(dim)
        for i, digit in enumerate(reversed(bin(state)[2:])):
            binary[-(i + 1)] = digit

        binary[target] = (binary[control] + binary[target]) % 2

        num = 0
        for i, comp in enumerate(reversed(binary)):
            num += (comp * (2**i))

        m[state, int(num)] = 1

    return m

In [87]:
dim = 3

def ZERO(dim):
    return np.array([[1]] + [[0]] * (2**dim - 1))

print(ZERO(dim))

I = np.diag(np.ones(2**dim))

I_1 = np.array([
    [1, 0],
    [0, 1]
])

CX = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
])

CXI = np.kron(CX, I_1)
ICX = np.kron(I_1, CX)

stab = CNOT(0, 2) @ CNOT(0, 1) @ H(0) @ ZERO(3)
stab

[[1]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]]


array([[0.70710678],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.70710678]])

In [99]:
dim = 9
vec = CNOT(6, 8) @ CNOT(6, 7) @ CNOT(3, 5) @ CNOT(3, 4) @ CNOT(0, 2) @ CNOT(0, 1) @ H(6) @ H(3) @ H(0) @ ZERO(dim)

def state(i, dim):
    binary = np.zeros(dim)
    for i, digit in enumerate(reversed(bin(i)[2:])):
        binary[-(i + 1)] = digit
    binary = [str(int(n)) for n in binary]
    return '|' + ''.join(binary) + '⟩'
    

for i, val in enumerate(vec):
    if val != 0:
        print(state(i, dim))


|000000000⟩
|000000111⟩
|000111000⟩
|000111111⟩
|111000000⟩
|111000111⟩
|111111000⟩
|111111111⟩


In [ ]:
import copy
dim = 3

oss = CNOT(0, 2) @ CNOT(0, 1) @ H(0) @ ZERO(3)
s1 = Z(0) @ Z(1)
s2 = Z(1) @ Z(2)
for _ in range(20):
    ss = copy.deepcopy(oss)
    if np.random.random() < 0.1:
        bit = np.random.randint(3)
        ss = X(bit) @ ss
    m1 = int(np.sum(s1 @ ss) < 0)
    m2 = int(np.sum(s2 @ ss) < 0)
    lst = [m1, m2]
    print(lst)
    

[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[1, 1]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]
[0, 0]


In [193]:
dim = 9

oss = CNOT(6, 8) @ CNOT(6, 7) @ CNOT(3, 5) @ CNOT(3, 4) @ CNOT(0, 2) @ CNOT(0, 1) @ H(6) @ H(3) @ H(0) @ ZERO(dim)
s1 = Z([0, 1])
s2 = Z([1, 2])
s3 = Z([3, 4])
s4 = Z([4, 5])
s5 = Z([6, 7])
s6 = Z([7, 8])
s7 = X([0, 1, 2, 3, 4, 5])
s8 = X([3, 4, 5, 6, 7, 8])
for _ in range(20):
    ss = copy.deepcopy(oss)

    if np.random.random() < 0.1:
        bit = np.random.randint(9)
        ss = X(bit) @ ss

    if np.random.random() < 0.1:
        bit = np.random.randint(9)
        ss = Z(bit) @ ss
    # sumss = np.sum(ss)
    # m1 = int(np.sum(s1 @ ss) != sumss)
    # m2 = int(np.sum(s2 @ ss) != sumss)
    # m3 = int(np.sum(s3 @ ss) != sumss)
    # m4 = int(np.sum(s4 @ ss) != sumss)
    # m5 = int(np.sum(s5 @ ss) != sumss)
    # m6 = int(np.sum(s6 @ ss) != sumss)
    m1 = int(np.round(np.dot((s1 @ ss).T, ss)[0][0]) != 1.0)
    m2 = int(np.round(np.dot((s2 @ ss).T, ss)[0][0]) != 1.0)
    m3 = int(np.round(np.dot((s3 @ ss).T, ss)[0][0]) != 1.0)
    m4 = int(np.round(np.dot((s4 @ ss).T, ss)[0][0]) != 1.0)
    m5 = int(np.round(np.dot((s5 @ ss).T, ss)[0][0]) != 1.0)
    m6 = int(np.round(np.dot((s6 @ ss).T, ss)[0][0]) != 1.0)
    m7 = int(np.round(np.dot((s7 @ ss).T, ss)[0][0]) != 1.0)
    m8 = int(np.round(np.dot((s8 @ ss).T, ss)[0][0]) != 1.0)
    lst = [m1, m2, m3, m4, m5, m6, m7, m8]
    print(lst)
    

[0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 1, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
